In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\Coding projects\fpl-analytics-pipeline


In [2]:
from src.utils.db import DUCKDB_PATH
print(DUCKDB_PATH)

d:\Coding projects\fpl-analytics-pipeline\data/fpl_warehouse.duckdb


In [3]:
from src.utils.db import get_connection

con = get_connection()

In [6]:
con.sql("SHOW SCHEMAS")

┌───────────────┬─────────────┬─────────┐
│ database_name │ schema_name │ current │
│    varchar    │   varchar   │ boolean │
├───────────────┼─────────────┼─────────┤
│ fpl_warehouse │ main        │ true    │
│ fpl_warehouse │ raw         │ false   │
└───────────────┴─────────────┴─────────┘

In [7]:
con.sql("SHOW TABLES FROM raw")

┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ events  │
│ players │
│ teams   │
└─────────┘

In [8]:
con.sql("SELECT COUNT(*) FROM raw.players")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          626 │
└──────────────┘

In [4]:
con.execute("DESCRIBE raw.players").fetchdf()

,column_name,column_type,null,key,default,extra
0,can_transact,BOOLEAN,YES,None,None,None
1,can_select,BOOLEAN,YES,None,None,None
2,chance_of_playing_next_round,DOUBLE,YES,None,None,None
3,chance_of_playing_this_round,DOUBLE,YES,None,None,None
4,code,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
104,selected_rank,BIGINT,YES,None,None,None
105,selected_rank_type,BIGINT,YES,None,None,None
106,starts_per_90,DOUBLE,YES,None,None,None
107,clean_sheets_per_90,DOUBLE,YES,None,None,None


In [5]:
con.execute("SELECT * FROM raw.players LIMIT 5").fetchdf()

,can_transact,can_select,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,price_change_percent,...,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,clean_sheets_per_90,defensive_contribution_per_90
0,True,True,NaN,NaN,154561,0,0,0,0,10.6,...,1,42,3,45,3,6,1,1.0,1.0,0.0
1,True,True,NaN,NaN,109745,0,0,0,0,-79.1,...,9,509,66,509,66,452,51,0.0,0.0,0.0
2,True,True,NaN,NaN,437495,0,0,0,0,-79.4,...,16,535,68,535,68,533,59,0.0,0.0,0.0
3,True,True,NaN,NaN,226597,0,0,0,0,-19.1,...,1,33,11,35,12,8,2,1.0,1.0,7.0
4,True,True,0.0,0.0,445122,0,0,0,0,-75.7,...,3,619,207,619,207,434,153,0.0,0.0,0.0


In [6]:
con.execute("SELECT COUNT(*) AS row_count FROM raw.players").fetchdf()

,row_count
0,626


In [4]:
player_columns = [
    "id",
    "web_name",
    "first_name",
    "second_name",
    "team",
    "element_type",
    "now_cost",
    "total_points",
    "form",
    "points_per_game",
    "selected_by_percent",
    "goals_scored",
    "assists",
    "clean_sheets",
    "minutes",
    "bonus",
    "yellow_cards",
    "red_cards",
    "transfers_in_event",
    "transfers_out_event",
    "value_season",
]

con.execute("""
    SELECT
        id,
        web_name,
        first_name,
        second_name,
        team,
        element_type,
        now_cost,
        total_points,
        form,
        points_per_game,
        selected_by_percent,
        goals_scored,
        assists,
        clean_sheets,
        minutes,
        bonus,
        yellow_cards,
        red_cards,
        transfers_in_event,
        transfers_out_event,
        value_season
    FROM raw.players
    LIMIT 10
""").fetchdf()

,id,web_name,first_name,second_name,team,element_type,now_cost,total_points,form,points_per_game,...,goals_scored,assists,clean_sheets,minutes,bonus,yellow_cards,red_cards,transfers_in_event,transfers_out_event,value_season
0,1,Raya,David,Raya Martín,1,1,60,12,6.0,6.0,...,0,0,2,180,0,0,0,72492,80393,2.0
1,2,Arrizabalaga,Kepa,Arrizabalaga Revuelta,1,1,50,0,0.0,0.0,...,0,0,0,0,0,0,0,276,499,0.0
2,3,Meslier,Illan,Meslier,1,1,50,0,0.0,0.0,...,0,0,0,0,0,0,0,62,190,0.0
3,4,Gabriel,Gabriel,dos Santos Magalhães,1,2,80,13,6.5,6.5,...,0,0,2,180,0,1,0,20711,127587,1.6
4,5,J.Timber,Jurriën,Timber,1,2,65,0,0.0,0.0,...,0,0,0,0,0,0,0,274,1258,0.0
5,6,Saliba,William,Saliba,1,2,60,0,0.0,0.0,...,0,0,0,0,0,0,0,713,3093,0.0
6,7,Lewis-Skelly,Myles,Lewis-Skelly,1,3,55,6,3.0,3.0,...,0,0,2,160,0,0,0,3536,4724,1.1
7,8,Calafiori,Riccardo,Calafiori,1,2,56,20,10.0,10.0,...,0,2,2,170,2,0,0,131451,51639,3.6
8,9,Hincapie,Piero,Hincapié,1,2,54,1,0.5,1.0,...,0,0,0,9,0,0,0,2862,12311,0.2
9,10,White,Benjamin,White,1,2,55,18,9.0,9.0,...,0,1,2,180,3,0,0,50732,39913,3.3


In [5]:
con.execute("""
    SELECT
        element_type,
        COUNT(*) AS player_count
    FROM raw.players
    GROUP BY element_type
    ORDER BY element_type
""").fetchdf()

,element_type,player_count
0,1,70
1,2,207
2,3,276
3,4,73


In [6]:
con.execute("""
    SELECT
        COUNT(*) AS total_players,
        COUNT(*) FILTER (WHERE minutes > 0) AS players_with_minutes,
        COUNT(*) FILTER (WHERE minutes = 0) AS players_without_minutes
    FROM raw.players
""").fetchdf()

,total_players,players_with_minutes,players_without_minutes
0,626,364,262


In [11]:
df=con.execute("""
SELECT 
    id as player_id,
    web_name as player_name,
    first_name || ' ' || second_name as full_name,
    team AS team_id,
    element_type AS position_id,
    CASE element_type
        WHEN 1 THEN 'GKP'
        WHEN 2 THEN 'DEF'
        WHEN 3 THEN 'MID'
        WHEN 4 THEN 'FWD'
    END AS position,
    ROUND(now_cost/10.0,1) as price_millions,
    total_points,
    CAST(form as FLOAT) AS form_rating,
    ROUND(CAST(points_per_game as FLOAT),2) AS pints_per_game,
    CAST(selected_by_percent AS FLOAT) AS ownership_pct,
    goals_scored,
    assists,
    clean_sheets,
    minutes,
    bonus,
    yellow_cards,
    red_cards,
    transfers_in_event AS gw_transfers_in,
    transfers_out_event AS gw_transfers_out,
    CAST(value_season AS FLOAT) AS value_season

FROM raw.players
WHERE minutes > 0;""").fetchdf()
df.head(10)
len(df)
df["position"].value_counts()

position
MID    172
DEF    127
FWD     43
GKP     22
Name: count, dtype: int64

In [12]:
len(df)
df["position"].value_counts()
df["minutes"].min()

np.int64(1)

Stagin teams analysis

In [13]:
con.close()